# Sinh biểu đồ coherence / diversity / runtime / robustness — CafeBERT full benchmark

Notebook này đọc trực tiếp `benchmark/cafebert_full/reference/full_results.csv` (480 dòng thật: 4 corpus × 6 mô hình × 5 giá trị k × 4 seed, đã audit `PASS`) và vẽ 4 nhóm biểu đồ, lưới 2×2 theo 4 corpus tiếng Việt (benchmark chỉ dùng một encoder CafeBERT nên không có chiều encoder).

Cột dữ liệu: **Coherence** → `wec_in` · **Diversity** → `topic_diversity` · **Runtime** → `fit_seconds`/`pipeline_seconds` (KHÔNG gộp lẫn) · **Robustness** → `c_npmi` (chỉ số phụ).

**Chạy notebook**: cần `jupyter`/`ipykernel` trong `.venv`:
```powershell
.venv\Scripts\python.exe -m pip install jupyter ipykernel
```
Biểu đồ xuất ra `benchmark/cafebert_full/notebook_charts/`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def find_repo_root(marker: str = "benchmark/cafebert_full/reference/full_results.csv") -> Path:
    """Works whether the notebook's cwd is the repo root or benchmark/cafebert_full/."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(
        f"Khong tim thay '{marker}' tu {Path.cwd()} hay bat ky thu muc cha nao. "
        "Hay chay notebook nay tu repo root (E:/Development/NLP_S3) hoac tu benchmark/cafebert_full/."
    )


ROOT = find_repo_root()
REFERENCE_DIR = ROOT / "benchmark" / "cafebert_full" / "reference"
CSV_PATH = REFERENCE_DIR / "full_results.csv"
OUTPUT_DIR = ROOT / "benchmark" / "cafebert_full" / "notebook_charts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repo root: {ROOT}")
print(f"Doc du lieu tu: {CSV_PATH}")
print(f"Xuat bieu do vao: {OUTPUT_DIR}")

In [ ]:
# Cung tu vung mau/nhan voi benchmark/cafebert_full/generate_cafebert_full_report.py.

TOPIC_COUNTS = [10, 20, 30, 40, 50]
SEEDS = [11, 29, 42, 47]

MODEL_ORDER = ["s3_axial", "s3_angular", "s3_combined", "lda", "nmf", "bertopic_kmeans"]
MODEL_LABELS = {
    "s3_axial": "S\u00b3 axial",
    "s3_angular": "S\u00b3 angular",
    "s3_combined": "S\u00b3 combined",
    "lda": "LDA",
    "nmf": "NMF",
    "bertopic_kmeans": "BERTopic + UMAP + KMeans",
}
# Xanh duong/tim/xanh la = 3 bien the S3; nau/xam/do = 3 baseline. TAT CA cac
# duong ve dam, ro rang (khong lam mo baseline nua) -- phan biet S3 vs baseline
# bang do day net (linewidth) va thu tu ve len tren (zorder), xem plot_metric_grid.
COLORS = {
    "S\u00b3 axial": "#1d4ed8",
    "S\u00b3 angular": "#7c3aed",
    "S\u00b3 combined": "#15803d",
    "LDA": "#b45309",
    "NMF": "#475569",
    "BERTopic + UMAP + KMeans": "#dc2626",
}

CORPUS_ORDER = ["vietnamese-news", "visfd", "vi-medical", "vntc-it"]
CORPUS_LABELS = {
    "vietnamese-news": "Vietnamese-news",
    "visfd": "UIT-ViSFD",
    "vi-medical": "ViMedical Disease",
    "vntc-it": "VNTC-CNTT",
}

In [ ]:
frame = pd.read_csv(CSV_PATH)
frame["method"] = pd.Categorical(
    frame["model"].map(MODEL_LABELS), [MODEL_LABELS[m] for m in MODEL_ORDER], ordered=True
)
frame["corpus_label"] = pd.Categorical(
    frame["corpus"].map(CORPUS_LABELS), [CORPUS_LABELS[c] for c in CORPUS_ORDER], ordered=True
)

expected_rows = len(CORPUS_ORDER) * len(MODEL_ORDER) * len(SEEDS) * len(TOPIC_COUNTS)
assert len(frame) == expected_rows, f"Ky vong {expected_rows} dong, doc duoc {len(frame)}"
assert frame["status"].eq("ok").all(), "Co dong status != 'ok', kiem tra lai truoc khi ve bieu do"
print(f"OK: {len(frame)} dong, tat ca status=ok.")
frame.head(3)

## Hàm vẽ dùng chung

Tất cả đường vẽ **đậm, rõ nét** (không còn hiệu ứng mờ trên baseline). Cách phân biệt S³ với baseline:
1. **Độ dày nét**: S³ `linewidth=3.2`, baseline `linewidth=2.2` — S³ nổi hơn nhưng baseline vẫn đọc rõ.
2. **`zorder`**: S³ vẽ đè lên trên baseline khi hai đường chồng nhau.
3. **Marker to hơn cho S³** (`6.5` vs `5.5`).
4. Dải mean ± SD giữ mờ nhẹ (`alpha` thấp) để không làm rối các đường đậm.
5. **Font to hơn ~1.3 lần** (`font.size=13`, tiêu đề ô `15`, suptitle `17`).
6. Lưới subplot 2×2 theo corpus, 1 legend chung ở trên; thang log cho runtime.

In [ ]:
def plot_metric_grid(
    frame: pd.DataFrame,
    metric: str,
    ylabel: str,
    title: str,
    output_path: Path,
    log_scale: bool = False,
    s3_linewidth: float = 3.2,
    baseline_linewidth: float = 2.2,
    s3_line_alpha: float = 1.0,
    baseline_line_alpha: float = 0.95,
    s3_fill_alpha: float = 0.14,
    baseline_fill_alpha: float = 0.07,
    base_fontsize: int = 13,
) -> Path:
    """2x2 grid (1 subplot / corpus), 1 duong mau / mo hinh, dai mean+-SD.
    Moi duong deu dam, ro net; S3 day hon + ve len tren de van noi bat.
    Font to hon ~1.3 lan so voi mac dinh. Luu PNG va tra ve duong dan."""
    plt.style.use("seaborn-v0_8-whitegrid")
    plt.rcParams.update({"font.size": base_fontsize})
    fig, axes = plt.subplots(2, 2, figsize=(14.5, 9.8), dpi=180, sharex=True)

    for ax, corpus in zip(axes.flat, CORPUS_ORDER, strict=True):
        part = frame.loc[frame["corpus"] == corpus]
        aggregate = (
            part.groupby(["method", "n_topics"], observed=True)[metric]
            .agg(["mean", "std"])
            .reset_index()
        )
        draw_order = [m for m in MODEL_ORDER if not m.startswith("s3_")] + [
            m for m in MODEL_ORDER if m.startswith("s3_")
        ]
        for model in draw_order:
            method = MODEL_LABELS[model]
            values = aggregate.loc[aggregate["method"] == method].sort_values("n_topics")
            is_s3 = model.startswith("s3_")
            line_alpha = s3_line_alpha if is_s3 else baseline_line_alpha
            fill_alpha = s3_fill_alpha if is_s3 else baseline_fill_alpha
            ax.plot(
                values["n_topics"],
                values["mean"],
                color=COLORS[method],
                alpha=line_alpha,
                marker="o",
                markersize=6.5 if is_s3 else 5.5,
                linewidth=s3_linewidth if is_s3 else baseline_linewidth,
                label=method,
                zorder=3 if is_s3 else 2,
            )
            sd = values["std"].fillna(0)
            ax.fill_between(
                values["n_topics"],
                values["mean"] - sd,
                values["mean"] + sd,
                color=COLORS[method],
                alpha=fill_alpha,
                zorder=1,
            )
        if log_scale:
            ax.set_yscale("log")
        ax.set_title(CORPUS_LABELS[corpus], loc="left", fontweight="bold", fontsize=base_fontsize + 2)
        ax.set_xticks(TOPIC_COUNTS)
        ax.set_xlabel("So topic (k)", fontsize=base_fontsize)
        ax.set_ylabel(ylabel, fontsize=base_fontsize)
        ax.tick_params(labelsize=base_fontsize - 1)
        ax.spines[["top", "right"]].set_visible(False)

    handles_by_label = dict(zip(*axes.flat[0].get_legend_handles_labels()[::-1]))
    ordered_labels = [MODEL_LABELS[m] for m in MODEL_ORDER]
    handles = [handles_by_label[label] for label in ordered_labels]
    fig.legend(
        handles, ordered_labels, loc="upper center", ncol=3,
        bbox_to_anchor=(0.5, 1.05), frameon=False, fontsize=base_fontsize,
    )
    fig.suptitle(title, y=1.12, fontsize=base_fontsize + 4, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    fig.savefig(output_path, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Da luu: {output_path}")
    return output_path

## 1. Coherence (WEC-in)

Cosine trung bình giữa các cặp top-term trong từng topic (Word2Vec huấn luyện trên chính corpus). Càng cao càng tốt.

In [ ]:
plot_metric_grid(
    frame,
    metric="wec_in",
    ylabel="WEC-in",
    title="Coherence (WEC-in) theo so topic, tren 4 corpus tieng Viet (CafeBERT)",
    output_path=OUTPUT_DIR / "topic_coherence.png",
)

## 2. Diversity

Tỉ lệ top-term không trùng lặp trên toàn bộ topic. Giá trị cao **không** đồng nghĩa coherence cao — đọc cùng biểu đồ coherence ở trên.

In [ ]:
plot_metric_grid(
    frame,
    metric="topic_diversity",
    ylabel="Topic diversity",
    title="Diversity theo so topic, tren 4 corpus tieng Viet (CafeBERT)",
    output_path=OUTPUT_DIR / "topic_diversity.png",
)

## 3. Runtime — fit-only vs pipeline (KHÔNG gộp lẫn)

- **`fit_seconds`** (fit-only, warm): chỉ thời gian fit mô hình sau khi embedding/CountVectorizer đã sẵn sàng — so sánh tốc độ thuật toán.
- **`pipeline_seconds`**: cộng thêm chi phí encode CafeBERT (S³/BERTopic) hoặc CountVectorizer (LDA/NMF) — không phải ablation encoder.

Cả hai ở thang log vì các phương pháp chênh nhau hàng chục/hàng trăm lần.

In [ ]:
plot_metric_grid(
    frame,
    metric="fit_seconds",
    ylabel="Fit-only, warm (giay, thang log)",
    title="Runtime fit-only theo so topic, tren 4 corpus tieng Viet (CafeBERT)",
    output_path=OUTPUT_DIR / "runtime_fit_only.png",
    log_scale=True,
)

In [ ]:
plot_metric_grid(
    frame,
    metric="pipeline_seconds",
    ylabel="Pipeline cold-reference (giay, thang log)",
    title="Runtime pipeline (bieu dien + fit) theo so topic, tren 4 corpus tieng Viet",
    output_path=OUTPUT_DIR / "runtime_pipeline.png",
    log_scale=True,
)

## 4. Robustness (C_NPMI)

Gensim C_NPMI — chỉ số coherence phụ (đồng xuất hiện từ), dùng kiểm tra không đồng thuận với WEC-in, **không** dùng chọn winner.

In [ ]:
plot_metric_grid(
    frame,
    metric="c_npmi",
    ylabel="C_NPMI (robustness, khong dung chon winner)",
    title="Robustness (C_NPMI) theo so topic, tren 4 corpus tieng Viet (CafeBERT)",
    output_path=OUTPUT_DIR / "robustness_c_npmi.png",
)

## Phụ lục: số ô WEC-in mà một biến thể S³ thắng

In [ ]:
def count_wec_wins(frame: pd.DataFrame, seed_scope: list[int]) -> tuple[int, int, dict[str, int]]:
    scoped = frame.loc[frame["seed"].isin(seed_scope)]
    total = 0
    s3_wins = 0
    corpus_wins = {corpus: 0 for corpus in CORPUS_ORDER}
    for (corpus, _seed, _k), group in scoped.groupby(["corpus", "seed", "n_topics"], observed=True):
        max_score = group["wec_in"].max()
        winners = set(group.loc[group["wec_in"].eq(max_score), "model"])
        total += 1
        if any(model.startswith("s3_") for model in winners):
            s3_wins += 1
            corpus_wins[corpus] += 1
    return s3_wins, total, corpus_wins


s3_wins, total_cells, corpus_wins = count_wec_wins(frame, SEEDS)
print(f"S3 dan dau WEC-in trong {s3_wins}/{total_cells} o corpus x seed x k.")
for corpus in CORPUS_ORDER:
    print(f"  {CORPUS_LABELS[corpus]:20s}: {corpus_wins[corpus]}/20")

## Dùng biểu đồ trong `report/paper.tex`

Copy PNG vào `report/figures/` với tên riêng của nhóm (không đè lên ảnh tái hiện từ paper gốc):
```
cp benchmark/cafebert_full/notebook_charts/topic_coherence.png report/figures/vn_benchmark_coherence.png
cp benchmark/cafebert_full/notebook_charts/runtime_fit_only.png report/figures/vn_benchmark_runtime.png
```